# Cointegration Strategy Testing Framework



This notebook tests baskets for cointegration sustainability and mean reversion speed.



## Testing Pipeline (with Data Splitting to Prevent Overfitting):

1. **Load price data** from parquet files

2. **Split data chronologically**:

   - Cluster data: Most recent 20% (1.2-0.8 quantile) - for basket generation

   - Cointegration data: Middle 60% (0.8-0.2 quantile) - for cointegration testing

   - Half-life data: Oldest 20% (0.2-0.0 quantile) - for mean reversion testing

3. **Generate candidate baskets** using cluster data (prevents overfitting)

4. **Test initial cointegration** using cointegration data (auto-deduplicates overlapping baskets)

5. **Filter by sustainability** using cointegration data (rolling windows + discrete periods)

6. **Filter by mean reversion speed** using half-life data (prevents data leakage)

7. **Output validated baskets** for strategy deployment



## Why Data Splitting?

- **Prevents overfitting**: Clustering on same data as testing causes false positives

- **Prevents look-ahead bias**: Using future data to select baskets

- **Prevents data leakage**: Testing on data used for training

- **Realistic validation**: Tests baskets on unseen data


In [4]:
import sys
from pathlib import Path
import logging
from datetime import datetime, timedelta, timezone

# Find workspace root
current = Path().resolve()
while current.name != 'hku-datawork' and current.parent != current:
    current = current.parent
workspace_root = current if current.name == 'hku-datawork' else Path('/home/leo/GitHub_Clones/hku_stuff/hku-datawork')
sys.path.insert(0, str(workspace_root))

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Import modules
from hypothesis_testing.cointegration.data_loader import load_price_data
from hypothesis_testing.cointegration.data_split import split_data_chronologically
from hypothesis_testing.cointegration.basket_generator import generate_baskets_clustering
from hypothesis_testing.cointegration.utils_parallel import test_baskets_cointegration_parallel
from hypothesis_testing.cointegration.filter_sustainability import filter_baskets_sustainability
from hypothesis_testing.cointegration.filter_mean_reversion import filter_baskets_mean_reversion
from hypothesis_testing.cointegration.deduplicate_baskets import filter_overlapping_baskets

logger.info("Imports completed")

2025-11-06 13:41:41,580 - __main__ - INFO - Imports completed


## Configuration

Set data paths, timeframes, and filter thresholds.

In [5]:
# Configuration
DATA_DIR = Path("/workspace-hku/hku-data/test_data")  # Use absolute path to actual data location
TIMEFRAME = '15m'
PRICE_TYPE = 'mark'  # 15m only available as 'mark', 1h only as 'index'
SYMBOLS = None  # None = auto-discover all symbols
# Discover available symbols from parquet files in data directory
pattern = f"*_{PRICE_TYPE}_{TIMEFRAME}_*.parquet"
parquet_files = list(DATA_DIR.glob(pattern))
available_symbols = set()
for f in parquet_files:
    # Pattern: perpetual_{SYMBOL}_mark_15m_...
    parts = f.stem.split('_')
    if len(parts) >= 2:
        symbol = parts[1]  # Extract symbol (e.g., "BTC" or "1000BONK")
        available_symbols.add(symbol)

AVAILABLE_SYMBOLS = sorted(list(available_symbols))
print(f"Found {len(AVAILABLE_SYMBOLS)} symbols available in {DATA_DIR}")

# Basket generation
MIN_BASKET_SIZE = 2
MAX_BASKET_SIZE = 4
N_CLUSTERS = 5


# Parallel processing
MAX_WORKERS = None  # None = use CPU count

# Bars per day (depends on timeframe)
BARS_PER_DAY = 96


Found 116 symbols available in /workspace-hku/hku-data/test_data


## Step 1: Load Price Data

Load historical price data from parquet files.

In [6]:
# Load 15-minute mark price data from existing parquet files
# Data is already available in DATA_DIR - no download needed
logger.info(f"Loading data from parquet files in {DATA_DIR}")
logger.info(f"Timeframe: {TIMEFRAME}, Price type: {PRICE_TYPE}")

# Load data using the existing data loader (loads from parquet files)
# SYMBOLS=None means it will discover and load all symbols automatically
price_data = load_price_data(
    data_dir=DATA_DIR,
    years_back=1.2,  # Load only last 2 years
    symbols=None,  # Auto-discover all symbols from parquet files
    timeframe=TIMEFRAME,
    price_type=PRICE_TYPE,
    max_workers=MAX_WORKERS
)

print(f"  Symbols: {len(price_data.columns)}")
print(f"  Timestamps: {len(price_data):,}")
print(f"  Date range: {price_data.index.min()} to {price_data.index.max()}")


2025-11-06 13:41:50,095 - __main__ - INFO - Loading data from parquet files in /workspace-hku/hku-data/test_data
2025-11-06 13:41:50,097 - __main__ - INFO - Timeframe: 15m, Price type: mark
2025-11-06 13:41:50,102 - hypothesis_testing.cointegration.data_loader - INFO - Loading data from 116 parquet files for 116 symbols...
2025-11-06 13:41:51,395 - hypothesis_testing.cointegration.data_loader - INFO - Aligning timestamps across all symbols...
2025-11-06 13:41:52,113 - hypothesis_testing.cointegration.data_loader - INFO - Loaded 34939 timestamps for 104 symbols
2025-11-06 13:41:52,115 - hypothesis_testing.cointegration.data_loader - INFO - Date range: 2024-09-13 12:15:00+00:00 to 2025-09-12 10:45:00+00:00


  Symbols: 104
  Timestamps: 34,939
  Date range: 2024-09-13 12:15:00+00:00 to 2025-09-12 10:45:00+00:00


## Step 2: Split Data Chronologically

**CRITICAL**: Split data to prevent overfitting and data leakage:
- **Cluster data** (1.2-0.8): Most recent 20% - used for basket generation
- **Cointegration data** (0.8-0.2): Middle 60% - used for cointegration testing
- **Half-life data** (0.2-0.0): Oldest 20% - used for mean reversion testing

This ensures we don't test on the same data we used to generate baskets.

In [7]:
# Split data chronologically to prevent overfitting
splits = split_data_chronologically(
    price_data,
    cluster_split=(1.2, 0.8),      # Most recent 20% for clustering
    cointegration_split=(0.8, 0.2), # Middle 60% for cointegration testing
    half_life_split=(0.2, 0.0)      # Oldest 20% for half-life testing
)

cluster_data = splits['cluster_data']
cointegration_data = splits['cointegration_data']
half_life_data = splits['half_life_data']



2025-11-06 13:41:53,714 - hypothesis_testing.cointegration.data_split - INFO - Split data chronologically:
2025-11-06 13:41:53,716 - hypothesis_testing.cointegration.data_split - INFO -   Cluster: 6987 samples (1.2-80%) from 2025-07-01 16:15:00+00:00 to 2025-09-12 10:45:00+00:00
2025-11-06 13:41:53,718 - hypothesis_testing.cointegration.data_split - INFO -   Cointegration: 20964 samples (80%-20%) from 2024-11-25 07:15:00+00:00 to 2025-07-01 16:00:00+00:00
2025-11-06 13:41:53,719 - hypothesis_testing.cointegration.data_split - INFO -   Half-life: 6988 samples (20%-0%) from 2024-09-13 12:15:00+00:00 to 2024-11-25 07:00:00+00:00


## Step 3: Generate Candidate Baskets

Generate candidate baskets using clustering on **cluster data only** (prevents overfitting).

In [8]:
# Extract base symbols (remove _close suffix)

base_symbols = [col.replace('_close', '') for col in price_data.columns]



candidate_baskets = generate_baskets_clustering(

    price_data=cluster_data,  # Use cluster_data, not price_data!

    n_clusters=N_CLUSTERS,

    min_basket_size=MIN_BASKET_SIZE,

    max_basket_size=MAX_BASKET_SIZE,

    parallel=True,

    max_workers=MAX_WORKERS

)



logger.info(f"Generated {len(candidate_baskets)} candidate baskets")

print(f"Sample baskets: {candidate_baskets[:5]}")

2025-11-06 13:41:56,185 - hypothesis_testing.cointegration.basket_generator - INFO - Limited cluster size from 20 to 17 symbols (estimated 6175 -> ~4029 combinations)
2025-11-06 13:41:56,382 - hypothesis_testing.cointegration.basket_generator - INFO - Limited cluster size from 80 to 17 symbols (estimated 1795199 -> ~4029 combinations)
2025-11-06 13:41:56,940 - __main__ - INFO - Generated 6393 candidate baskets


Sample baskets: [['TRX', 'SUN'], ['PENDLE', 'QNT'], ['PENDLE', 'STG'], ['PENDLE', 'BAND'], ['PENDLE', 'CRV']]


In [28]:
# Optional: Analyze optimal cluster count
analysis = analyze_cluster_quality(
    price_data=cluster_data,  # Use cluster_data
    min_clusters=2,
    max_clusters=15,
    min_basket_size=MIN_BASKET_SIZE,  # Match actual generation
    max_basket_size=MAX_BASKET_SIZE,  # Match actual generation (critical!)
    max_combinations_per_cluster=4000
)

print(f"\nRecommended cluster count: {analysis['recommended_n_clusters']}")
print(
    f"  Total baskets at optimal: "
    f"{analysis['total_baskets'][analysis['n_clusters'].index(analysis['recommended_n_clusters'])]:,}"
)
print(
    f"  Clusters hitting limit: "
    f"{analysis['clusters_limited'][analysis['n_clusters'].index(analysis['recommended_n_clusters'])]}"
)

# Use recommended if different from current
if analysis['recommended_n_clusters'] != N_CLUSTERS:
    print(
        f"\n⚠️  Consider using n_clusters={analysis['recommended_n_clusters']} "
        f"instead of {N_CLUSTERS}"
    )



Recommended cluster count: 10
  Total baskets at optimal: 7,494
  Clusters hitting limit: 3

⚠️  Consider using n_clusters=10 instead of 5


## Step 4: Test Initial Cointegration

Test all candidate baskets for cointegration using Johansen test on **cointegration data** (separate from cluster data).
Automatically deduplicates baskets with >50% symbol overlap (keeps lowest p-value).

In [11]:
cointegrated_baskets = test_baskets_cointegration_parallel(
    price_data=cointegration_data,  # Use cointegration_data, not price_data!
    candidate_baskets=candidate_baskets,
    max_workers=MAX_WORKERS,
    batch_size=100,
    overlap_threshold=0.5,
    prefer_lower_pvalue=True,
)

logger.info(
    f"Found {len(cointegrated_baskets)} unique cointegrated baskets "
    f"(after dedup) out of {len(candidate_baskets)} candidates"
)
print(
    f"Cointegration rate (after dedup): "
    f"{len(cointegrated_baskets)/len(candidate_baskets):.1%}"
)


2025-11-06 13:09:44,817 - hypothesis_testing.cointegration.utils_parallel - INFO - Processed 10/182 batches (1000 baskets), found 508 cointegrated so far
2025-11-06 13:09:51,258 - hypothesis_testing.cointegration.utils_parallel - INFO - Processed 20/182 batches (2000 baskets), found 1365 cointegrated so far
2025-11-06 13:09:56,496 - hypothesis_testing.cointegration.utils_parallel - INFO - Processed 30/182 batches (3000 baskets), found 2360 cointegrated so far
2025-11-06 13:10:01,635 - hypothesis_testing.cointegration.utils_parallel - INFO - Processed 40/182 batches (4000 baskets), found 3357 cointegrated so far
2025-11-06 13:10:06,982 - hypothesis_testing.cointegration.utils_parallel - INFO - Processed 50/182 batches (5000 baskets), found 4354 cointegrated so far
2025-11-06 13:10:11,913 - hypothesis_testing.cointegration.utils_parallel - INFO - Processed 60/182 batches (6000 baskets), found 5335 cointegrated so far
2025-11-06 13:10:16,920 - hypothesis_testing.cointegration.utils_parall

Cointegration rate: 93.4%


2025-11-06 13:11:08,601 - hypothesis_testing.cointegration.deduplicate_baskets - INFO - Deduplicated baskets: 71 kept, 16897 removed (overlap threshold: 50%)
2025-11-06 13:11:08,602 - __main__ - INFO - Deduplicated 71 candidates


## Step 5: Filter by Sustainability

Test cointegration persistence across rolling windows and discrete periods using **cointegration data**.
This ensures the cointegration relationship is stable over time (not just in-sample).

In [15]:
# Sustainability filter thresholds
PERSISTENCE_THRESHOLD = 0.7  # 70% of windows/periods must pass
WINDOW_DAYS = 30
STEP_DAYS = 10
PERIOD_DAYS = 30

# Mean reversion filter thresholds
HALF_LIFE_THRESHOLD_DAYS = 30.0  # Maximum half-life in days

sustainable_baskets = filter_baskets_sustainability(
    baskets=cointegrated_baskets,
    price_data=cointegration_data,  # Use cointegration_data
    persistence_threshold=PERSISTENCE_THRESHOLD,
    window_days=WINDOW_DAYS,
    step_days=STEP_DAYS,
    period_days=PERIOD_DAYS,
    bars_per_day=BARS_PER_DAY,
    use_rolling=True,
    use_discrete=True,
    max_workers=MAX_WORKERS
)

logger.info(
    f"Filtered to {len(sustainable_baskets)} sustainable baskets "
    f"(from {len(cointegrated_baskets)} cointegrated after dedup)"
)

# Display sample results
if sustainable_baskets:
    sample = sustainable_baskets[0]
    print(f"
Sample sustainable basket: {sample['basket']}")
    print(
        f"Rolling windows persistence: "
        f"{sample['sustainability']['rolling_windows']['persistence_ratio']:.1%}"
    )
    print(
        f"Discrete periods persistence: "
        f"{sample['sustainability']['discrete_periods']['persistence_ratio']:.1%}"
    )


2025-11-06 13:13:48,767 - hypothesis_testing.cointegration.filter_sustainability - INFO - Testing 71 baskets for sustainability (parallelized at basket level with 38 workers)...
2025-11-06 13:14:04,966 - hypothesis_testing.cointegration.filter_sustainability - INFO - Processed 3/71 baskets, found 3 sustainable so far
2025-11-06 13:14:05,086 - hypothesis_testing.cointegration.filter_sustainability - INFO - Processed 6/71 baskets, found 6 sustainable so far
2025-11-06 13:14:05,154 - hypothesis_testing.cointegration.filter_sustainability - INFO - Processed 9/71 baskets, found 9 sustainable so far
2025-11-06 13:14:05,236 - hypothesis_testing.cointegration.filter_sustainability - INFO - Processed 12/71 baskets, found 12 sustainable so far
2025-11-06 13:14:05,316 - hypothesis_testing.cointegration.filter_sustainability - INFO - Processed 15/71 baskets, found 15 sustainable so far
2025-11-06 13:14:05,371 - hypothesis_testing.cointegration.filter_sustainability - INFO - Processed 18/71 baskets


Sample sustainable basket: ['AVAX', 'RENDER', 'ETH', 'ETC']
Rolling windows persistence: 100.0%
Discrete periods persistence: 100.0%


## Step 6: Filter by Mean Reversion Speed

Filter for baskets that mean revert quickly using **half-life data** (separate from cointegration data).
This tests whether the spread reverts fast enough to be profitable on unseen data.

In [ ]:
fast_mean_reverting_baskets = filter_baskets_mean_reversion(
    baskets=sustainable_baskets,
    half_life_data=half_life_data,  # Use half_life_data (prevents data leakage)
    half_life_threshold_days=HALF_LIFE_THRESHOLD_DAYS,
    bars_per_day=BARS_PER_DAY,
    max_workers=MAX_WORKERS
)

logger.info(f"Filtered to {len(fast_mean_reverting_baskets)} fast mean-reverting baskets "
           f"(from {len(sustainable_baskets)} sustainable)")

# Display sample results
if fast_mean_reverting_baskets:
    sample = fast_mean_reverting_baskets[0]
    print(f"\nSample fast mean-reverting basket: {sample['basket']}")
    print(f"Half-life: {sample['mean_reversion']['half_life_days']:.1f} days")
    print(f"ADF p-value: {sample['mean_reversion']['adf_p_value']:.4f}")
    print(f"Is stationary: {sample['mean_reversion']['is_stationary']}")

## Step 7: Save Validated Baskets

Save the proven baskets for strategy deployment.

In [ ]:
import json

# Prepare baskets for saving (convert numpy arrays to lists)
validated_baskets_output = []
for basket_result in fast_mean_reverting_baskets:
    output = {
        'basket': basket_result['basket'],
        'eigenvector': basket_result['eigenvector'].tolist(),
        'johansen_p_value': basket_result['johansen_result']['p_value'],
        'johansen_trace_stat': float(basket_result['johansen_result']['trace_stat']),
        'sustainability_rolling': basket_result['sustainability']['rolling_windows']['persistence_ratio'],
        'sustainability_discrete': basket_result['sustainability']['discrete_periods']['persistence_ratio'],
        'half_life_days': float(basket_result['mean_reversion']['half_life_days']),
        'adf_p_value': float(basket_result['mean_reversion']['adf_p_value']),
        'is_stationary': basket_result['mean_reversion']['is_stationary']
    }
    validated_baskets_output.append(output)

# Save to JSON
output_file = Path("validated_baskets.json")
with open(output_file, 'w') as f:
    json.dump({
        'timestamp': datetime.now(timezone.utc).isoformat(),
        'config': {
            'timeframe': TIMEFRAME,
            'price_type': PRICE_TYPE,
            'persistence_threshold': PERSISTENCE_THRESHOLD,
            'half_life_threshold_days': HALF_LIFE_THRESHOLD_DAYS
        },
        'baskets': validated_baskets_output
    }, f, indent=2)

logger.info(f"Saved {len(validated_baskets_output)} validated baskets to {output_file}")

# Display summary
print(f"
{'='*60}")
print("VALIDATION SUMMARY")
print(f"{'='*60}")
print(f"Total candidate baskets: {len(candidate_baskets)}")
print(f"Cointegrated baskets (deduplicated): {len(cointegrated_baskets)} ({len(cointegrated_baskets)/len(candidate_baskets):.1%})")
print(f"Sustainable baskets: {len(sustainable_baskets)} ({len(sustainable_baskets)/len(cointegrated_baskets):.1%} of cointegrated)")
print(f"Fast mean-reverting baskets: {len(fast_mean_reverting_baskets)} ({len(fast_mean_reverting_baskets)/len(sustainable_baskets):.1%} of sustainable)")
print(f"{'='*60}")
print(f"
Validated baskets ready for strategy deployment!")
print(f"Output file: {output_file}")
